In [1]:
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
import pandas as pd

In [2]:
manga_fits = fits.open('~/Downloads/drpall-v3_1_1.fits')
manga_fits.info()

Filename: /Users/hyp0515/Downloads/drpall-v3_1_1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       6   ()      
  1  MANGA         1 BinTableHDU    209   11273R x 99C   [K, 32A, 32A, 32A, 32A, 32A, 32A, 32A, 32A, 32A, 32A, D, D, D, D, D, D, D, K, D, K, D, D, 53A, K, 32A, K, D, D, D, D, D, D, D, D, D, D, D, K, K, K, D, D, D, D, K, K, K, K, 32A, K, 19A, K, K, K, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, K, 6A, K, K, D, D, 7D, 7D, 7D, D, D, D, D, 7D, D, D, 7D, 7D, 7D, 7D, D, D, D, D, 7D, 7D]   
  2  MASTAR        1 BinTableHDU    209   35394R x 99C   [K, 32A, 32A, 32A, 32A, 32A, 32A, 32A, 32A, 32A, 32A, D, D, D, D, D, D, D, K, D, K, D, D, 53A, K, 32A, K, D, D, D, D, D, D, D, D, D, D, D, K, K, K, D, D, D, D, K, K, K, K, 32A, K, 19A, K, K, K, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, D, K, K, K, 6A, K, K, D, D, 7D, 7D, 7D, D, D, D, D, 7D, D, D, 7D, 7D, 7D, 7D, D, D, D, D, 7D, 7D]   


In [3]:
manga_fits['MANGA'].columns

ColDefs(
    name = 'plate'; format = 'K'
    name = 'ifudsgn'; format = '32A'
    name = 'plateifu'; format = '32A'
    name = 'mangaid'; format = '32A'
    name = 'versdrp2'; format = '32A'
    name = 'versdrp3'; format = '32A'
    name = 'verscore'; format = '32A'
    name = 'versutil'; format = '32A'
    name = 'versprim'; format = '32A'
    name = 'platetyp'; format = '32A'
    name = 'srvymode'; format = '32A'
    name = 'objra'; format = 'D'
    name = 'objdec'; format = 'D'
    name = 'ifuglon'; format = 'D'
    name = 'ifuglat'; format = 'D'
    name = 'ifura'; format = 'D'
    name = 'ifudec'; format = 'D'
    name = 'ebvgal'; format = 'D'
    name = 'nexp'; format = 'K'
    name = 'exptime'; format = 'D'
    name = 'drp3qual'; format = 'K'
    name = 'bluesn2'; format = 'D'
    name = 'redsn2'; format = 'D'
    name = 'harname'; format = '53A'
    name = 'frlplug'; format = 'K'
    name = 'cartid'; format = '32A'
    name = 'designid'; format = 'K'
    name = 'cenra'; format

In [4]:
manga_table = Table(manga_fits['MANGA'].data)
names = [name for name in manga_table.colnames if len(manga_table[name].shape) <= 1]
manga_fits_df = manga_table[names].to_pandas()
manga_fits_df = manga_fits_df[['mangaid', 'objra', 'objdec']]
manga_fits_df.head(10)


,mangaid,objra,objdec
0,1-48157,133.371091,57.598425
1,1-48188,133.685670,57.480250
2,1-55648,136.017160,57.092329
3,1-55616,133.989967,57.677968
4,1-55784,136.751375,57.451437
5,1-55567,133.330028,57.041155
6,1-48201,134.193923,56.786747
7,1-48111,132.465647,57.143728
8,1-48136,132.912768,57.107424
9,1-55612,134.591499,57.684965


In [5]:
from scipy.spatial import KDTree

# Create a KDTree for fast spatial queries on the Manga dataframe
# Note: We only need to do this once.
manga_coords = np.c_[manga_fits_df['objra'], manga_fits_df['objdec']]
manga_tree = KDTree(manga_coords)

def find_all_matches(spectra_df, manga_df, manga_tree, search_radius_deg=0.001):
    """
    Finds the closest match in manga_df for each object in spectra_df using a KDTree.
    """
    # Get coordinates from the spectra dataframe
    spectra_coords = np.c_[spectra_df['RA'], spectra_df['DEC']]
    
    # Query the tree to find the nearest neighbor for each spectrum object
    # It returns the distance and the index of the neighbor in manga_df
    distances, indices = manga_tree.query(spectra_coords, k=1)

    # Convert search radius from degrees to the squared distance used by the tree
    # This is a good approximation for small angular separations.
    max_distance = search_radius_deg
    
    # Filter out matches that are outside the search radius
    valid_indices_mask = distances < max_distance

    # Get the indices from manga_df for valid matches
    matched_manga_indices = indices[valid_indices_mask]

    # Create a new dataframe with the matched morphological data
    matched_df = manga_df.iloc[matched_manga_indices].copy()

    # Get the corresponding data from the spectra dataframe to merge
    matched_spectra_df = spectra_df[valid_indices_mask].copy()
    
    # Reset indices to allow for a clean merge
    matched_df.reset_index(drop=True, inplace=True)
    matched_spectra_df.reset_index(drop=True, inplace=True)
    
    # Combine the dataframes side-by-side
    # This merges the original spectra info with the new morphology info
    result_df = pd.concat([matched_spectra_df, matched_df], axis=1)
    # result_df.drop(columns=['catalog_version', 'legacy_survey_data_release', 'brickid', 'objid', 'brickid'], errors='ignore', inplace=True)
    # result_df.drop(columns=['SPECTYPE', 'FLUX_G', 'FLUX_R', 'FLUX_Z', 'z_pipe'], errors='ignore', inplace=True)
    return result_df

In [ ]:
dps_fits = fits.open('./catalogs/all_catalog_test.fits')
dps_df = Table(dps_fits[1].data).to_pandas()

In [18]:
dps_df[['TARGETID', 'RA', 'DEC']].to_csv('./catalogs/dp_catalog_test.csv', index=False)

In [ ]:
# matched_df = find_all_matches(dps_df, manga_fits_df, manga_tree, search_radius_deg=0.001)
# matched_df.head(10)

,TARGETID,RA,DEC,Z,LOGM,LOGSFR,dv_r,dv_l,sigma_r,sigma_l,...,OIII5007_rank,NII6548_rank,Halpha_rank,NII6583_rank,SII6716_rank,SII6731_rank,dp_count,mangaid,objra,objdec
0,39627745847678361,202.715347,-1.778709,0.036098,10.488779,1.005706,122.657204,-31.675087,13.441554,84.307518,...,8,6,0,1,5,7,2,1-159093,202.715316,-1.778673
1,39627763941905330,201.215454,-1.035095,0.106646,10.880481,0.665281,200.181717,-86.357513,97.590508,178.735580,...,6,2,1,0,5,7,1,1-641156,201.215462,-1.035083
